In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import polars as pl

In [2]:
#1.path
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent

DATA_ROOT = PROJECT_ROOT / "tennis_data"

EXTRACT_ROOT = DATA_ROOT / "extracted"


print("PROJECT ROOT:", PROJECT_ROOT)
print("DATA ROOT:", DATA_ROOT)
print("EXTRACT ROOT:", EXTRACT_ROOT)

PROJECT ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis
DATA ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data
EXTRACT ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted


In [3]:
# 2. Find all Venue parquet files

venue_files = sorted(
    EXTRACT_ROOT.glob(
        "*/venue_*.parquet"
    )
)


print(
    "Number of Venue files:",
    len(venue_files)
)

Number of Venue files: 35423


In [4]:
# 3. Inspect one Venue parquet file


test_venue_df = pl.read_parquet(
    venue_files[0]
)


print(test_venue_df)


print("\nSchema:")
print(test_venue_df.schema)

shape: (1, 5)
┌──────────┬───────────┬───────────────┬──────────┬─────────────┐
│ match_id ┆ city      ┆ stadium       ┆ venue_id ┆ country     │
│ ---      ┆ ---       ┆ ---           ┆ ---      ┆ ---         │
│ i64      ┆ str       ┆ str           ┆ i64      ┆ str         │
╞══════════╪═══════════╪═══════════════╪══════════╪═════════════╡
│ 11974053 ┆ Groningen ┆ Martini Plaza ┆ 7324     ┆ Netherlands │
└──────────┴───────────┴───────────────┴──────────┴─────────────┘

Schema:
Schema({'match_id': Int64, 'city': String, 'stadium': String, 'venue_id': Int64, 'country': String})


In [5]:
# 4. Check schemas of all Venue files


schemas = Counter()


for file in venue_files:

    df = pl.read_parquet(file)

    schema_tuple = tuple(
        df.schema.items()
    )

    schemas[schema_tuple] += 1


print("Number of different schemas:",len(schemas))

Number of different schemas: 2


In [7]:
# 5.Display Venue schema summary

for i, (schema, count) in enumerate(schemas.items(), start=1):

    print("=" * 60)

    print(f"Schema {i}")
    print(f"Number of files: {count}")

    for column, dtype in schema:
        print(f"{column} -> {dtype}")

Schema 1
Number of files: 35264
match_id -> Int64
city -> String
stadium -> String
venue_id -> Int64
country -> String
Schema 2
Number of files: 159
match_id -> Int64
city -> String
stadium -> String
venue_id -> Int64
country -> Null


In [8]:
# 6.Compare data types across Venue schemas

column_dtypes = defaultdict(set)


for schema, count in schemas.items():

    for column, dtype in schema:

        column_dtypes[column].add(str(dtype))


for column, dtypes in column_dtypes.items():

    if len(dtypes) > 1:

        print(f"{column}: {dtypes}")

country: {'String', 'Null'}


In [9]:
# 7.Define standard schema for Venue dataset


venue_schema = {
    "match_id": pl.Int64,
    "city": pl.String,
    "stadium": pl.String,
    "venue_id": pl.Int64,
    "country": pl.String
}


print(venue_schema)

{'match_id': Int64, 'city': String, 'stadium': String, 'venue_id': Int64, 'country': String}


In [10]:
# 8.Read and process all Venue files


venue_frames = []


# Read each Venue parquet file

for file in venue_files:

    df = pl.read_parquet(file)

    snapshot_date = file.parent.name

    df = df.with_columns(
        pl.lit(snapshot_date)
        .str.strptime(pl.Date, "%Y%m%d")
        .alias("snapshot_date")
    )


    # Apply standard schema
    # strict=False keeps null values without errors

    df = df.cast(
        venue_schema,
        strict=False
    )


    venue_frames.append(df)


print(
    "Number of processed files:",
    len(venue_frames)
)

Number of processed files: 35423


In [11]:
# 9.Concatenate all Venue dataframes

# Combine all processed Venue dataframes vertically

venue_snapshot = pl.concat(
    venue_frames,
    how="vertical"
)


print("Final shape:")
print(venue_snapshot.shape)


print("\nSchema:")
print(venue_snapshot.schema)

Final shape:
(35423, 6)

Schema:
Schema({'match_id': Int64, 'city': String, 'stadium': String, 'venue_id': Int64, 'country': String, 'snapshot_date': Date})


In [12]:
# 10.Check missing values in Venue dataset


venue_snapshot.null_count()

match_id,city,stadium,venue_id,country,snapshot_date
u32,u32,u32,u32,u32,u32
0,0,0,0,159,0


In [14]:
# 11.Check for duplicate rows

duplicate_count = (
    venue_snapshot.height
    - venue_snapshot.unique().height
)


print("Number of duplicate rows:",duplicate_count)

Number of duplicate rows: 0


In [15]:
#12. Check match IDs with multiple snapshots


venue_snapshot_counts = (
    venue_snapshot
    .group_by("match_id")
    .agg(
        pl.col("snapshot_date")
        .n_unique()
        .alias("number_of_snapshots")
    )
)


multiple_venue_snapshots = (
    venue_snapshot_counts
    .filter(
        pl.col("number_of_snapshots") > 1
    )
)


print("Match IDs with multiple snapshots:", multiple_venue_snapshots.shape[0])


multiple_venue_snapshots.head(10)

Match IDs with multiple snapshots: 16239


match_id,number_of_snapshots
i64,u32
12042472,2
12077747,2
12199902,2
12027651,2
12061467,2
12168039,2
12079938,2
12108425,3
12124803,2


In [16]:
# 13.Check Venue information changes between snapshots


venue_changes = (
    venue_snapshot
    .group_by("match_id")
    .agg(
        pl.col("venue_id")
        .n_unique()
        .alias("different_venue_ids"),

        pl.col("city")
        .n_unique()
        .alias("different_cities"),

        pl.col("stadium")
        .n_unique()
        .alias("different_stadiums"),

        pl.col("country")
        .n_unique()
        .alias("different_countries")
    )
)


changed_venue_matches = (
    venue_changes
    .filter(
        (pl.col("different_venue_ids") > 1)
        |
        (pl.col("different_cities") > 1)
        |
        (pl.col("different_stadiums") > 1)
        |
        (pl.col("different_countries") > 1)
    )
)


print("Number of matches with Venue changes:",changed_venue_matches.shape[0])


changed_venue_matches.head(10)

Number of matches with Venue changes: 150


match_id,different_venue_ids,different_cities,different_stadiums,different_countries
i64,u32,u32,u32,u32
12165556,2,1,2,1
12060687,2,1,2,1
12061464,2,1,2,1
12165815,2,1,2,1
12165559,2,1,2,1
12086527,2,1,2,1
12165553,2,1,2,1
12086530,2,1,2,1
12165535,2,1,2,1


In [17]:
# 14.Inspect Venue changes for one match


match_id_example = 12165556


venue_snapshot.filter(
    pl.col("match_id") == match_id_example
)

match_id,city,stadium,venue_id,country,snapshot_date
i64,str,str,i64,str,date
12165556,"""Antalya""","""Court 21""",28266,"""Turkey""",2024-03-16
12165556,"""Antalya""","""MTA 3""",31365,"""Turkey""",2024-03-17
12165556,"""Antalya""","""MTA 3""",31365,"""Turkey""",2024-03-18


In [18]:
# 15.Check Venue ID consistency


venue_id_consistency = (
    venue_snapshot
    .group_by("venue_id")
    .agg(
        pl.col("stadium")
        .n_unique()
        .alias("unique_stadiums"),

        pl.col("city")
        .n_unique()
        .alias("unique_cities"),

        pl.col("country")
        .n_unique()
        .alias("unique_countries")
    )
)


inconsistent_venues = (
    venue_id_consistency
    .filter(
        (pl.col("unique_stadiums") > 1)
        |
        (pl.col("unique_cities") > 1)
        |
        (pl.col("unique_countries") > 1)
    )
)


print(
    "Venue IDs with inconsistent information:",
    inconsistent_venues.shape[0]
)


inconsistent_venues.head(10)

Venue IDs with inconsistent information: 0


venue_id,unique_stadiums,unique_cities,unique_countries
i64,u32,u32,u32


In [19]:
# 16.Save cleaned Venue dataset


clean_path = DATA_ROOT / "Data"

clean_path.mkdir(
    parents=True,
    exist_ok=True
)


venue_snapshot.write_parquet(
    clean_path / "venue_clean.parquet"
)


print("Saved successfully:")
print(clean_path / "venue_clean.parquet")

Saved successfully:
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/Data/venue_clean.parquet


In [24]:
# 17.Verify saved Venue dataset


venue_clean = pl.read_parquet(
    clean_path / "venue_clean.parquet"
)


print("Shape:")
print(venue_clean.shape)


print("\nSchema:")
print(venue_clean.schema)
venue_clean.head(20)

Shape:
(35423, 6)

Schema:
Schema({'match_id': Int64, 'city': String, 'stadium': String, 'venue_id': Int64, 'country': String, 'snapshot_date': Date})


match_id,city,stadium,venue_id,country,snapshot_date
i64,str,str,i64,str,date
11974053,"""Groningen""","""Martini Plaza""",7324,"""Netherlands""",2024-02-01
11974066,"""Vilnius""","""SEB Arena""",36345,"""Lithuania""",2024-02-01
11998445,"""Montpellier""","""Court Patrice Dominguez""",20333,"""France""",2024-02-01
11998446,"""Montpellier""","""Court Patrice Dominguez""",20333,"""France""",2024-02-01
11998447,"""Montpellier""","""Court Patrice Dominguez""",20333,"""France""",2024-02-01
…,…,…,…,…,…
11998672,"""Burnie""","""Ct 5""",20229,"""Australia""",2024-02-01
11998674,"""Burnie""","""Centre Court""",24474,"""Australia""",2024-02-01
11998675,"""Burnie""","""Centre Court""",24474,"""Australia""",2024-02-01
